# Imports

In [ ]:
import json

import pandas as pd
import numpy as np

In [ ]:
# The operating threshold is produced by 02_Modeling.ipynb — never hard-code it here,
# otherwise the dashboard silently drifts away from the evaluated model.
with open('../models/best_threshold.json') as f:
    best_threshold = json.load(f)['best_threshold']

df_train = pd.read_csv('../data/raw/cs-training.csv', index_col=0)
base_rate = df_train['SeriousDlqin2yrs'].mean()

print(f'Operating threshold (OOF F2): {best_threshold:.3f}')
print(f'Portfolio default rate:       {base_rate:.3f}')

In [ ]:
df_test = pd.read_csv('../data/raw/cs-test.csv', index_col=0)
df_test.index.name = 'Id'
df_test = df_test.drop(columns='SeriousDlqin2yrs')

df_prediction = pd.read_csv('../data/processed/kaggle_submission.csv', index_col=0)
df_merged = pd.merge(df_test, df_prediction, how='inner', on='Id')

# Flagged for manual review == the model's positive class, at the same threshold
# that was used to report precision/recall in the README.
df_merged['Predicted_class'] = (df_merged['Probability'] >= best_threshold).astype(int)

# Risk bands anchored on interpretable quantities rather than round numbers:
#   low    — risk below the portfolio average
#   medium — above average, but not enough to trigger a review
#   high   — at or above the operating threshold
df_merged['Risk'] = pd.cut(
    df_merged['Probability'],
    bins=[-np.inf, base_rate, best_threshold, np.inf],
    labels=['low', 'medium', 'high'],
)

df_merged['Risk'].value_counts(normalize=True).reindex(['low', 'medium', 'high'])

In [ ]:
df_merged.head()

In [ ]:
df_merged.to_excel('../data/processed/powerbi_dataset.xlsx', index=True)
print(f'Saved {len(df_merged):,} rows for Power BI.')